In [ ]:
df_news_final_project = pd.read_parquet('https://storage.googleapis.com/msca-bdp-data-open/news_final_project/news_final_project.parquet', engine='pyarrow')
df_news_final_project.shape

## Step 1: Basic Schema Handling

### Objective
Validate required columns and normalize core text fields.

### Operations
- Check required columns (`title` + `content`/`text`).
- Safely convert text fields to strings and trim whitespace.
- Remove rows where article body is missing or empty.

### Why it matters
This is the foundation for all later steps and prevents type/empty-text issues in regex cleaning and filtering.

### Output
A valid `articles_df` with non-empty article text and row counts after missing-value removal.


## Step 2: Remove HTML and Crawl Artifacts

### Objective
Remove web-crawl formatting noise while preserving semantic article content.

### Operations in `clean_text()`
- Decode HTML entities (for example `&amp;`, `&nbsp;`).
- Strip HTML tags (`<...>`).
- Remove inline URLs.
- Remove common boilerplate phrases (`subscribe`, `advertisement`, `read more`, etc.).
- Remove obvious JS/CSS fragments.
- Normalize repeated whitespace and extra line breaks.

### Why it matters
This step improves text quality and reduces non-semantic tokens that can distort keyword matching and NLP models.

### Output
Two cleaned columns: `clean_title` and `clean_content`, with empty cleaned rows removed.


## Step 3: Length Filtering

### Objective
Remove very short, low-information articles.

### Operations
- Compute `content_char_len` and `word_count` as quality indicators.
- Apply a minimum character threshold (for example `MIN_CONTENT_CHARS = 200`).

### Why it matters
Very short texts are often snippets, navigation fragments, or weak evidence for industry-impact analysis.

### Output
A higher-information subset plus length statistics for sanity checks.


## Step 4: Exact Deduplication

### Objective
Remove exact duplicate articles to avoid double-counting the same content.

### Operations
- Primary pass: deduplicate by `clean_content` (keep first).
- Optional stricter pass: deduplicate by `clean_title + clean_content`.

### Why it matters
News datasets commonly contain mirrored/reposted copies. Keeping them would bias company/industry frequency analysis.

### Output
A deduplicated dataset with row counts after exact duplicate removal.


## Step 5: Relevance Filtering

### Objective
Keep only AI/ML/Data Science-relevant articles and remove clearly off-topic content.

### Rule design
- `include_keywords`: AI, machine learning, deep learning, LLM, generative AI, NLP, computer vision, etc.
- `exclude_keywords`: sports, politics, weather, celebrity gossip, etc.
- Case-insensitive matching over `title + content`.

### Filtering logic
- Keep a row only if it matches at least one include keyword.
- Rows that only match exclude topics and do not show AI relevance are removed.

### Why it matters
This step aligns the corpus with the project objective: identifying AI impact on industries and companies.

### Output
A topic-focused corpus with row counts after relevance filtering.


## Final Output

### Final dataframe
The final output is `clean_articles_df` (or `final`) with the core schema:
- `article_id`
- `date`
- `title`
- `text`

### Reproducibility
The notebook prints row counts at each stage (raw -> missing-value drop -> short-text filter -> dedup -> relevance -> final) for transparent reporting.

### Next step
Use this output for entity extraction (industry/company), impact direction classification, mechanism tagging, and topic modeling.


In [ ]:
clean_articles_df.to_parquet("clean_articles_step1.parquet", index=False)